<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/Function_Calling_%EC%97%B0%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install langchain openai tiktoken yfinance langchain_community langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import os



In [25]:
import yfinance as yf
import random

def get_stock_price(symbol):

  ticker = yf.Ticker(symbol)
  todays_data = ticker.history(period = '1d')

  return todays_data['Close'].iloc[0]

def get_lottery_number(max_num):
  return random.randint(0,max_num)

In [26]:
from pydantic import BaseModel, Field
from langchain.tools import BaseTool

class StockPriceCheckInput(BaseModel):
  stockticker : str = Field(description = "Ticker symbol for stock or Index")

class StockPriceTool(BaseTool):
  name : str = 'get_stock_ticker_price'
  description : str = '주식 가격을 알때 쓰는 툴입니다. 주식 티코 (종목 코드)를 입력받아 yfinance API를 실행합니다.'
  args_schema: type[BaseModel] = StockPriceCheckInput

  def _run(self, stockticker : str):
    price_response = get_stock_price(stockticker)
    return price_response

  # def _arun(self, stockticker: str):
  #   raise NotImplementedError("This tool does not support async")


class LotteryMaxCheckInput(BaseModel):
  max_number : int = Field(description = "max number for lottery random integer number creator.")

class LotteryNumberTool(BaseTool):
  name : str = 'get_lottery_number_tool'
  description : str = '로또 정답을 추출할 때 쓰는 툴입니다. 정수 값을 입력받아 0부터 입력된 정수값 사이 랜덤 숫자를 생성합니다.'
  args_schema : type[BaseModel] = LotteryMaxCheckInput

  def _run(self, max_num : int):
    answer = get_lottery_number(max_num)
    return answer

LLM 호출

In [105]:
#1. LLM 도구와 합치기
from langchain_openai import ChatOpenAI
#Langchain의 여러 Message 형식
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

model = ChatOpenAI(model="gpt-4o")
tools = [StockPriceTool(), LotteryNumberTool()]

model_w_tools = model.bind_tools(tools)

In [87]:
#2. LLM Tool Call 파라미터 반환
ai_message = model_w_tools.invoke('구글 주식 가격 알려줘')

In [88]:
#3. Tool Call 파라미터로부터 Tool Message 구성요소 추출
tool_name = ai_message.tool_calls[0]['name']
tool_args = ai_message.tool_calls[0]['args']
tool_call_id = ai_message.tool_calls[0]['id']

tool_result = tools[0].invoke(tool_args)

In [90]:
#4. Tool Message 구성
tool_message = ToolMessage(
    name = tool_name,
    content = str(tool_result),
    tool_call_id = tool_call_id
)

In [115]:
#5. LLM 재호출
message_history = [HumanMessage(content = '구글 주식 가격 알려줘'),
                   ai_message, #AIMessage()
                   tool_message] #ToolMessage()
model.invoke(message_history)

AIMessage(content='현재 구글의 주식 가격은 약 307.04달러입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 52, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_44968754fe', 'id': 'chatcmpl-DI6rJDoO7ZZFCjbcCWH62X3z96GUt', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cdb78-9fe7-7a80-ae7f-c04c86244550-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 17, 'total_tokens': 69, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [103]:
type(message_history[0]),type(message_history[1]),type(message_history[2])
#HumanMessage, AIMessage, ToolMessage

(langchain_core.messages.human.HumanMessage,
 langchain_core.messages.ai.AIMessage,
 langchain_core.messages.tool.ToolMessage)